Mục tiêu: Check A/B validity của Dataset 

Ngày phân tích: 2026-08-06



Step 1: Reload Data

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from pathlib import Path

DATA_PATH = Path("../data/cookie_cats.csv")

df = pd.read_csv(
    DATA_PATH,
    dtype={
        "userid": "int64",
        "version": "string",
        "sum_gamerounds": "int64",
        "retention_1": "bool",
        "retention_7": "bool",
    },
)

print(f"Loaded {len(df):,} rows")

Loaded 90,189 rows


Step 2: SRM Check

In [ ]:
# SRM check: kiểm tra xem các biến phân loại (categorical) có phân phối đồng đều giữa các nhóm A/B hay không
# Chi-square goodness-of-fit test: https://en.wikipedia.org/wiki/Chi-squared_test
# H0: split thực tế = split design (tức là phân phối đồng đều giữa các nhóm A/B)
# H1: split thực tế != split design (tức là phân phối không đồng đều giữa các nhóm A/B)
threshold = 0.001  # significance level -> nếu p-value < threshold thì reject H0, tức là phân phối không đồng đều giữa các nhóm A/B

# Bước 1: Đếm observations theo các biến phân loại
observed = df["version"].value_counts().sort_index()
print("Observed distribution:")
print(observed)
print()

n_total = observed.sum()
observed_array = observed.values

# Bước 2: Tính expected distribution theo split design (tức là phân phối đồng đều giữa các nhóm A/B)
expected_ratio = np.array([0.5, 0.5])  # expected ratio for A/B split
expected_array = n_total * expected_ratio
print(f"Expected (50/50): [{expected_array[0]:,.0f}, {expected_array[1]:,.0f}]")
print(f"Observed: [{observed_array[0]:,.0f}, {observed_array[1]:,.0f}]")
print(f"Difference: [{observed_array[0] - expected_array[0]:,.0f}, {observed_array[1] - expected_array[1]:,.0f}]")
print()

# Bước 3: Tính chi-square goodness-of-fit test
chi2_stat, p_value = stats.chisquare(f_obs=observed_array, f_exp=expected_array)
print(f"Chi-square statistic: {chi2_stat:.3f}")
print(f"P-value: {p_value:.3f}")
print()

# Bước 4: Kết luận
srm_threshold = 0.001
if p_value < srm_threshold:
    print(f"Reject H0: p-value < {srm_threshold} → phân phối không đồng đều giữa các nhóm A/B")
    print("Randomization check failed: có thể có vấn đề với quá trình randomization hoặc data integrity, stop việc phân tích")
else:
    print(f"Fail to reject H0: p-value >= {srm_threshold} → phân phối đồng đều giữa các nhóm A/B")
    print("Randomization check passed: phân phối đồng đều giữa các nhóm A/B, có thể tiếp tục phân tích")

# Bước 5: Hiển thị tỷ lệ
pct_gate_30 = observed_array[0] / n_total * 100
pct_gate_40 = observed_array[1] / n_total * 100
print(f"\nSplit ratio: gate 30: {pct_gate_30:.2f}% | gate 40: {pct_gate_40:.2f}%")

Observed distribution:
version
gate_30    44700
gate_40    45489
Name: count, dtype: Int64

Expected (50/50): [45,094, 45,094]
Observed: [44,700, 45,489]
Difference: [-394, 394]

Chi-square statistic: 6.902
P-value: 0.009

Fail to reject H0: p-value >= 0.001 → phân phối đồng đều giữa các nhóm A/B
Randomization check passed: phân phối đồng đều giữa các nhóm A/B, có thể tiếp tục phân tích

Split ratio: gate 30: 49.56% | gate 40: 50.44%


Step 3: SRM by segment 

In [ ]:
# SRM by segment - Kiểm tra sự mất cân bằng giữa các group
def srm_by_segment(sub_df: pd.DataFrame, segment_name: str, threshold: float = 0.001):
    observed = sub_df["version"].value_counts().sort_index()

# Nếu nhỏ hơn 20 observations thì bỏ qua vì không đủ dữ liệu để kiểm tra SRM
    if observed.sum() < 20:
        return {
            "segment": segment_name,
            "n_total": len(sub_df),
            "n_gate_30": int(observed.get("gate_30", 0)),
            "n_gate_40": int(observed.get("gate_40", 0)),
            "p_value": None,
            "verdict": "Not enough data (<20 observations)",
        }

    n_total = observed.sum()
    expected_ratio = np.array([0.5, 0.5]) 
    expected = n_total * expected_ratio
    observed_array = np.array([observed.get("gate_30", 0), observed.get("gate_40", 0)])

    chi2,p = stats.chisquare(f_obs=observed_array, f_exp=expected)
    verdict = "Fail to reject H0: phân phối đồng đều" if p >= threshold else "Reject H0: phân phối không đồng đều"

    return {
        "segment": segment_name,
        "n_total": int(n_total),
        "n_gate_30": int(observed.get("gate_30", 0)),
        "n_gate_40": int(observed.get("gate_40", 0)),
        "pct_gate_30": round(observed_array[0] / n_total * 100, 2),
        "p_value": round(p, 4),
        "verdict": verdict,
    }
    


# Segment 1: overall (đã làm ở Cell 2, chạy lại cho consistency)
seg_all = srm_by_segment(df, "ALL users")

# Segment 2: users có sum_gamerounds = 0 (đã install nhưng chưa chơi round nào)
seg_zero = srm_by_segment(df[df["sum_gamerounds"] == 0], "0 rounds only")

# Segment 3: users 0 rounds nhưng retention_1 = True 
seg_zero_r1 = srm_by_segment(
    df[(df["sum_gamerounds"] == 0) & (df["retention_1"] == True)],
    "0 rounds & retention_1=True",
)

# Segment 4: users 0 rounds nhưng retention_7 = True
seg_zero_r7 = srm_by_segment(
    df[(df["sum_gamerounds"] == 0) & (df["retention_7"] == True)],
    "0 rounds & retention_7=True",
)

# Segment 5: users vượt qua level 30 (chạm gate cũ)
seg_past_gate30 = srm_by_segment(
    df[df["sum_gamerounds"] >= 30],
    "sum_gamerounds >= 30 (past gate_30)",
)

# Segment 6: users vượt qua level 40 (chạm gate mới)
seg_past_gate40 = srm_by_segment(
    df[df["sum_gamerounds"] >= 40],
    "sum_gamerounds >= 40 (past gate_40)",
)

# Tổng hợp thành DataFrame
srm_by_segment = pd.DataFrame([
    seg_all, seg_zero, seg_zero_r1, seg_zero_r7, seg_past_gate30, seg_past_gate40
])

print("srm_by_segment:")
print(srm_by_segment.to_string(index=False))

srm_by_segment:
                            segment  n_total  n_gate_30  n_gate_40  pct_gate_30  p_value                               verdict
                          ALL users    90189      44700      45489        49.56   0.0086 Fail to reject H0: phân phối đồng đều
                      0 rounds only     3994       1937       2057        48.50   0.0576 Fail to reject H0: phân phối đồng đều
        0 rounds & retention_1=True       87         41         46        47.13   0.5919 Fail to reject H0: phân phối đồng đều
        0 rounds & retention_7=True       29         16         13        55.17   0.5775 Fail to reject H0: phân phối đồng đều
sum_gamerounds >= 30 (past gate_30)    33269      16656      16613        50.06   0.8136 Fail to reject H0: phân phối đồng đều
sum_gamerounds >= 40 (past gate_40)    27393      13566      13827        49.52   0.1148 Fail to reject H0: phân phối đồng đều


Step 4: Retentio distribution (chưa test, chỉ mô tả)

In [ ]:
# Descriptive: retention rate 2 nhóm — CHƯA TEST significance

retention_summary = df.groupby("version").agg(
    n_users=("userid", "count"),
    retention_1_rate=("retention_1", "mean"),
    retention_7_rate=("retention_7", "mean"),
    mean_rounds=("sum_gamerounds", "mean"),
    median_rounds=("sum_gamerounds", "median"),
).round(4)

print("Retention & engagement by version (descriptive only):")
print(retention_summary)
print()

# Delta absolute
r1_gate30 = retention_summary.loc["gate_30", "retention_1_rate"]
r1_gate40 = retention_summary.loc["gate_40", "retention_1_rate"]
r7_gate30 = retention_summary.loc["gate_30", "retention_7_rate"]
r7_gate40 = retention_summary.loc["gate_40", "retention_7_rate"]

print(f"Retention D1: {(r1_gate40 - r1_gate30)*100:+.3f} pp  "
      f"(gate_40 vs gate_30)")
print(f"Retention D7: {(r7_gate40 - r7_gate30)*100:+.3f} pp  "
      f"(gate_40 vs gate_30)")
print()


Retention & engagement by version (descriptive only):
         n_users  retention_1_rate  retention_7_rate  mean_rounds  \
version                                                             
gate_30    44700            0.4482            0.1902      52.4563   
gate_40    45489            0.4423            0.1820      51.2988   

         median_rounds  
version                 
gate_30           17.0  
gate_40           16.0  

Retention D1: -0.590 pp  (gate_40 vs gate_30)
Retention D7: -0.820 pp  (gate_40 vs gate_30)

